In [1]:
import duckdb

from pathlib import Path

code, product_name, generic_name, created_datetime, last_modified_datetime, countries_tags, brands, categories_tags, categories, main_category, completeness, quantity, product_quantity

'0001', 'Product A', NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-01-02 00:00:00', 'en:france', 'BrandA', 'en:dog-food', 'Dog food', 'dog-food', 0.8, '1kg', 1.0
'0001', 'Product A (older)', NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2025-12-01 00:00:00', 'en:france', 'BrandA', 'en:dog-food', 'Dog food', 'dog-food', 0.5, '1kg', 1.0
'0002', NULL, NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-01-01 00:00:00', 'en:germany', 'BrandB', 'en:cat-food', 'Cat food', 'cat-food', 0.6, '500g', 0.5
NULL, 'No Code Product', NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-01-01 00:00:00', 'en:spain', 'BrandC', 'en:cat-food', 'Cat food', 'cat-food', 0.7, '2kg', 2.0


In [9]:
def _write_sample_bronze(path):
    con = duckdb.connect()
    con.execute(f"""
        COPY (
            SELECT * FROM (VALUES
                ('0001', 'Product A', NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-01-02 00:00:00', 'en:france', 'BrandA', 'en:dog-food', 'Dog food', 'dog-food', 0.8, '1kg', 1.0),
                ('0001', 'Product A (older)', NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2025-12-01 00:00:00', 'en:france', 'BrandA', 'en:dog-food', 'Dog food', 'dog-food', 0.5, '1kg', 1.0),
                ('0002', NULL, NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-01-01 00:00:00', 'en:germany', 'BrandB', 'en:cat-food', 'Cat food', 'cat-food', 0.6, '500g', 0.5),
                (NULL, 'No Code Product', NULL, TIMESTAMP '2026-01-01 00:00:00', TIMESTAMP '2026-01-01 00:00:00', 'en:spain', 'BrandC', 'en:cat-food', 'Cat food', 'cat-food', 0.7, '2kg', 2.0)
            ) AS t(code, product_name, generic_name, created_datetime, last_modified_datetime, countries_tags, brands, categories_tags, categories, main_category, completeness, quantity, product_quantity)
        ) TO '{path}' (FORMAT PARQUET)
    """)
    con.close()

In [10]:
_write_sample_bronze(Path("/tmp/test_bronze.parquet"))

In [11]:
result = duckdb.sql("SELECT * FROM '/tmp/test_bronze.parquet'").fetchall()
for row in result:
    print(row)

('0001', 'Product A', None, datetime.datetime(2026, 1, 1, 0, 0), datetime.datetime(2026, 1, 2, 0, 0), 'en:france', 'BrandA', 'en:dog-food', 'Dog food', 'dog-food', Decimal('0.8'), '1kg', Decimal('1.0'))
('0001', 'Product A (older)', None, datetime.datetime(2026, 1, 1, 0, 0), datetime.datetime(2025, 12, 1, 0, 0), 'en:france', 'BrandA', 'en:dog-food', 'Dog food', 'dog-food', Decimal('0.5'), '1kg', Decimal('1.0'))
('0002', None, None, datetime.datetime(2026, 1, 1, 0, 0), datetime.datetime(2026, 1, 1, 0, 0), 'en:germany', 'BrandB', 'en:cat-food', 'Cat food', 'cat-food', Decimal('0.6'), '500g', Decimal('0.5'))
(None, 'No Code Product', None, datetime.datetime(2026, 1, 1, 0, 0), datetime.datetime(2026, 1, 1, 0, 0), 'en:spain', 'BrandC', 'en:cat-food', 'Cat food', 'cat-food', Decimal('0.7'), '2kg', Decimal('2.0'))


In [12]:
from datetime import date


In [13]:
from src.processing.silver_layer import silver_layer

In [16]:
dest_path=silver_layer(
    bronze_path=Path("/tmp/test_bronze.parquet"),
    dest_dir=Path("/tmp/test_silver"),
    run_date=date(2026, 7, 1)
)

print(dest_path)

/tmp/test_silver/2026-07-01/products.parquet


In [17]:
result=duckdb.sql(f"SELECT * FROM '{dest_path}'").fetchall()
for row in result:
    print(row)

('0001', 'Product A', None, datetime.datetime(2026, 1, 1, 8, 0, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), datetime.datetime(2026, 1, 2, 8, 0, tzinfo=<DstTzInfo 'Asia/Manila' PST+8:00:00 STD>), 'en:france', 'BrandA', 'en:dog-food', 'Dog food', 'dog-food', Decimal('0.8'), '1kg', Decimal('1.0'))
